## 1. Preprocessing

In [1]:
from preprocessing import run_preprocessing
results= run_preprocessing(
        place_name="Kasarani, Nairobi, Kenya",
        gee_project="causal-bus-404912",
        buffer_m=30,
        gee_buildings_asset="projects/sat-io/open-datasets/MSBuildings/Kenya",
        gee_buildings_local_path="kasarani_buildings_export.geojson",
        raster_local_path="kasarani_sentinel_full_stack.tif",
        output_dir="./output/kasarani",
    )

## 2. Modelling with Random Forest

In [3]:
from modeling import run_modeling

results = run_modeling(
    results["training_table"], results["feature_cols"], results["output_dir"],
    label_col="encroachment", model_name="encroachment_rf", task_type="binary",
)

## 3. Evaluation

In [4]:
# View precision, recall, f1, and roc_auc
print("Model Evaluation Metrics:")
for metric, score in results["metrics"].items():
    print(f"  {metric}: {score:.4f}")

Model Evaluation Metrics:
  precision: 0.0949
  recall: 0.4745
  f1: 0.1582
  roc_auc: 0.7691


In [5]:
import pandas as pd

# Extract feature importances directly from the fitted model
model = results["model"]

importance_df = pd.DataFrame({
    "Feature": model.feature_names_in_,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\nFeature Importances:")
print(importance_df.to_string(index=False))


Feature Importances:
 Feature  Importance
    ndbi    0.182079
mean_B11    0.168033
   mndwi    0.147845
 mean_B2    0.113576
    ndvi    0.105526
 mean_B8    0.102657
 mean_B3    0.095341
 mean_B4    0.084945


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualizations for each task (Building Detection, Land Cover, Encroachment)
for task_name in task_keys:
    task_res = results[task_name]
    model = task_res["model"]
    df = task_res["training_table"]
    feature_cols = list(model.feature_names_in_)
    label_col = [c for c in df.columns if c not in feature_cols and c != "split"][0]
    
    test_df = df[df["split"] == "test"]
    X_test = test_df[feature_cols]
    y_test = test_df[label_col]
    y_pred = model.predict(X_test)
    
    unique_classes = sorted(list(set(y_test) | set(y_pred)))
    class_labels = [str(c) for c in unique_classes]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. Feature Importances
    importance_df = pd.DataFrame({
        "Feature": feature_cols,
        "Importance": model.feature_importances_
    }).sort_values(by="Importance", ascending=False)
    
    sns.barplot(data=importance_df, x="Importance", y="Feature", palette="viridis", ax=axes[0])
    axes[0].set_title(f"Feature Importances - {task_name.replace('_', ' ').title()}")
    
    # 2. Confusion Matrix Heatmap (Works for Multiclass and Binary)
    cm = confusion_matrix(y_test, y_pred, labels=unique_classes)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
                xticklabels=class_labels, yticklabels=class_labels, ax=axes[1])
    axes[1].set_title(f"Confusion Matrix - {task_name.replace('_', ' ').title()}")
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("Actual")
    
    plt.tight_layout()
    plt.show()

Classification Report:
                 precision    recall  f1-score   support

Non-Encroaching       0.98      0.85      0.91     11108
    Encroaching       0.09      0.47      0.16       373

       accuracy                           0.84     11481
      macro avg       0.54      0.66      0.53     11481
   weighted avg       0.95      0.84      0.88     11481

Confusion Matrix:
[[9420 1688]
 [ 196  177]]
